In [1]:
!pip install imbalanced-learn --break-system-packages

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 3, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 159.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 75.7 MB/s eta 0:00:00
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 2.2.0
    Uninstalling threadpoolctl-2.2.0:
      Successfully uninstalled threadpoolctl-2.2.0
  Attempting uninstall: joblib
    Found existing installation: joblib 1.2.0
    Uninstalling joblib-1.2.0:
      Successfully uninstalled joblib-1.2.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which i

In [3]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
 
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix
)
 
print("=" * 60)
print("XGBoost + SMOTE — Supervised Fraud Classifier")
print("=" * 60)
 
mlflow.set_experiment("FraudDetection_XGBoost")


StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 5, Finished, Available, Finished, False)

XGBoost + SMOTE — Supervised Fraud Classifier


<Experiment: artifact_location='sds://onelakecentralindia.pbidedicated.windows.net/c9a67f7c-cad6-4156-8afd-be7ae3dea097/e5ace7b9-7a5a-4ae0-8457-1e7e1005aa38', creation_time=1778215615006, experiment_id='e5ace7b9-7a5a-4ae0-8457-1e7e1005aa38', last_update_time=1778215615006, lifecycle_stage='active', name='FraudDetection_XGBoost', tags={}>

## Load Features

In [4]:
FEAT_COLS = [f"V{i}" for i in range(1, 29)] + ["Amount", "hour_of_day"]
TARGET    = "Class"
 
df_pd = spark.read.format("delta").table("silver_features") \
             .select([TARGET] + FEAT_COLS) \
             .toPandas()
 
X = df_pd[FEAT_COLS].fillna(0)
y = df_pd[TARGET]
 
pos_rate = y.mean()
print(f"Fraud rate         : {pos_rate:.4%}")
print(f"Imbalance ratio    : 1:{int(1/pos_rate)}")
print(f"Total transactions : {len(X):,}")
print(f"Fraud transactions : {y.sum():,}")

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 6, Finished, Available, Finished, False)

Fraud rate         : 0.1727%
Imbalance ratio    : 1:578
Total transactions : 284,807
Fraud transactions : 492


In [5]:
display(df_pd)

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a60b930-520b-4aa5-b5b3-4beb5055da2d)

## Train / Test Split 

In [6]:
# Stratified to maintain 0.17% fraud in both splits
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain: {len(X_tr):,}  |  Test: {len(X_te):,}")
print(f"Fraud in train: {y_tr.sum():,}  |  Fraud in test: {y_te.sum():,}")

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 8, Finished, Available, Finished, False)


Train: 227,845  |  Test: 56,962
Fraud in train: 394  |  Fraud in test: 98


## SMOTE — Synthetic Minority Oversampling 

In [7]:
print(f"\nBefore SMOTE — Train fraud rate: {y_tr.mean():.4%}")
 
smote = SMOTE(
    sampling_strategy = 0.1,   # target ratio: 10% fraud (1:10)
    k_neighbors       = 5,     # nearest neighbours for interpolation
    random_state      = 42,
)
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
 
print(f"After SMOTE  — Train fraud rate: {y_tr_sm.mean():.4%}")
print(f"After SMOTE  — Total train rows: {len(X_tr_sm):,}")
print(f"Synthetic fraud samples added  : {y_tr_sm.sum() - y_tr.sum():,}")

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 9, Finished, Available, Finished, False)


Before SMOTE — Train fraud rate: 0.1729%
After SMOTE  — Train fraud rate: 9.0909%
After SMOTE  — Total train rows: 250,196
Synthetic fraud samples added  : 22,351


 At 1:578 imbalance, class_weight='balanced' alone gives the model
 insufficient fraud signal. SMOTE generates synthetic fraud
 samples to bring the training ratio to 10% fraud (1:10).
 IMPORTANT: SMOTE is applied ONLY to training data — never test data.

 How SMOTE works:
 1. For each fraud transaction, find its k nearest fraud neighbours
 2. Generate a new synthetic sample by interpolating between
    the real sample and one of its neighbours
 3. The synthetic sample is a realistic fraud transaction,
    not just a duplicate

## XGBoost Parameters

In [9]:
# scale_pos_weight is still used but much lower than raw 1:578
# because SMOTE already balanced training data to 1:10
pos_after_smote = y_tr_sm.mean()
scale_pw = (1 - pos_after_smote) / pos_after_smote  # 10
 
XGB_PARAMS = dict(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 5,
    scale_pos_weight  = scale_pw,
    eval_metric       = "aucpr",  # AP is better than AUC for imbalance
    random_state      = 42,
    n_jobs            = -1,
    verbosity         = 0,
)

print(scale_pw)

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 11, Finished, Available, Finished, False)

10.00004396570675


## Train + MLflow

In [10]:
with mlflow.start_run(run_name="XGBoost_SMOTE_v1"):
 
    model = XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_tr_sm, y_tr_sm,
        eval_set            = [(X_te, y_te)],
        early_stopping_rounds = 50,
        verbose             = False
    )
 
    preds_prob = model.predict_proba(X_te)[:, 1]
    preds_cls  = (preds_prob >= 0.50).astype(int)
 
    auc = roc_auc_score(y_te, preds_prob)
    ap  = average_precision_score(y_te, preds_prob)
 
    # Confusion matrix on test set
    cm = confusion_matrix(y_te, preds_cls)
    tn, fp, fn, tp = cm.ravel()
 
    # Feature importance
    fi = pd.DataFrame({
        "feature":    FEAT_COLS,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)
 
    fi_path = "/tmp/feature_importance_fraud.csv"
    fi.to_csv(fi_path, index=False)
 
    # Log everything
    mlflow.log_params(XGB_PARAMS)
    mlflow.log_param("smote_sampling_strategy", 0.1)
    mlflow.log_param("smote_k_neighbors",        5)
    mlflow.log_param("train_rows_after_smote",   len(X_tr_sm))
    mlflow.log_metric("AUC_ROC",        round(auc, 4))
    mlflow.log_metric("Avg_Precision",  round(ap,  4))
    mlflow.log_metric("best_iteration", model.best_iteration)
    mlflow.log_metric("TP",  int(tp))
    mlflow.log_metric("FP",  int(fp))
    mlflow.log_metric("TN",  int(tn))
    mlflow.log_metric("FN",  int(fn))
    mlflow.log_artifact(fi_path, "feature_importance")
    mlflow.sklearn.log_model(
        model, "xgb_fraud",
        registered_model_name="FraudXGBoost"
    )
 
    print(f"\n{'='*50}")
    print("XGBOOST + SMOTE RESULTS")
    print("="*50)
    print(f"AUC-ROC:          {auc:.4f}")
    print(f"Avg Precision:    {ap:.4f}")
    print(f"Best iteration:   {model.best_iteration}")
    print(f"\nConfusion matrix (test set, threshold=0.50):")
    print(f"  True Positives  (fraud caught)   : {tp:>6,}")
    print(f"  False Positives (false alerts)   : {fp:>6,}")
    print(f"  True Negatives  (correct legit)  : {tn:>6,}")
    print(f"  False Negatives (missed fraud)   : {fn:>6,}")
    print(f"\n  Precision : {tp/(tp+fp):.4f}")
    print(f"  Recall    : {tp/(tp+fn):.4f}")
    print(f"\nTop 10 features by XGBoost importance:")
    print(fi.head(10).to_string(index=False))

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 13, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
2026/05/08 04:55:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp3hn9e2cm/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.8.0', 'cloudpickle==2.2.1']. Set logging level to DEBUG to see the full traceback. 
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'FraudXGBoost'.



XGBOOST + SMOTE RESULTS
AUC-ROC:          0.9928
Avg Precision:    0.8701
Best iteration:   495

Confusion matrix (test set, threshold=0.50):
  True Positives  (fraud caught)   :     84
  False Positives (false alerts)   :     31
  True Negatives  (correct legit)  : 56,833
  False Negatives (missed fraud)   :     14

  Precision : 0.7304
  Recall    : 0.8571

Top 10 features by XGBoost importance:
feature  importance
    V14    0.488773
    V10    0.150141
     V4    0.059298
    V12    0.043223
     V3    0.022161
     V8    0.016464
    V17    0.015592
     V1    0.014506
 Amount    0.013113
    V25    0.012659


## Threshold Analysis

In [11]:
print("\nThreshold sensitivity (test set):")
print(f"{'Threshold':>10} {'TP':>6} {'FP':>8} {'FN':>6} {'Precision':>10} {'Recall':>8}")
print("-"*55)
for thr in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    pred_t = (preds_prob >= thr).astype(int)
    cm_t   = confusion_matrix(y_te, pred_t).ravel()
    tn_t, fp_t, fn_t, tp_t = cm_t
    prec = tp_t/(tp_t+fp_t) if (tp_t+fp_t)>0 else 0
    rec  = tp_t/(tp_t+fn_t) if (tp_t+fn_t)>0 else 0
    print(f"  ≥{thr:.2f}     {tp_t:>5,}  {fp_t:>8,}  {fn_t:>5,}  "
          f"{prec:>8.3f}  {rec:>8.3f}")

StatementMeta(, cdec4355-7abb-4553-ac93-82e9c8e7311a, 15, Finished, Available, Finished, False)


Threshold sensitivity (test set):
 Threshold     TP       FP     FN  Precision   Recall
-------------------------------------------------------
  ≥0.30        86        46     12     0.652     0.878
  ≥0.40        84        39     14     0.683     0.857
  ≥0.50        84        31     14     0.730     0.857
  ≥0.60        84        21     14     0.800     0.857
  ≥0.70        84        14     14     0.857     0.857
  ≥0.80        84        13     14     0.866     0.857
  ≥0.90        84         9     14     0.903     0.857
